# TP 1 - Optimisation Avancee d'un CNN sur Fashion-MNIST

**Module :** Deep Learning Avance & Applications
**Formation :** Master Bac+5 - Data Analytics & Big Data - Golden Collar
**Seance :** 1 - Optimisation avancee des reseaux neuronaux
**Duree estimee :** 3h

> Note : on utilise **Fashion-MNIST** (28x28, 10 classes) car il est leger et permet
> de comparer rapidement plusieurs techniques. Les idees se transposent telles quelles
> a CIFAR-10 ou a des images plus grandes.

---

## Objectifs du TP

1. **Partie A** - Fonctions de cout pour donnees desequilibrees : Cross-Entropy vs Focal Loss vs Weighted CE
2. **Partie B** - Normalisation : sans / BatchNorm / LayerNorm / GroupNorm
3. **Partie C** - Modele optimise (AdamW + He + BatchNorm + Label Smoothing + Augmentation) + ablation
4. **Partie D** - Analyse comparative et analyse d'erreurs

## Fil conducteur

Tout le TP repose sur **trois briques reutilisables** definies une seule fois :
`build_cnn(...)`, `train(...)`, `evaluate(...)`. On ne change qu'**un parametre a la fois**
pour isoler l'effet de chaque technique.

---
## 0. Configuration, donnees et briques reutilisables

In [ ]:
# Imports
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Reproductibilite
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))

In [ ]:
# Chargement et preparation de Fashion-MNIST
(X_full, y_full), (X_test, y_test) = fashion_mnist.load_data()

# Normalisation [0, 1] + dimension channel (28, 28) -> (28, 28, 1)
X_full = (X_full.astype('float32') / 255.0)[..., None]
X_test = (X_test.astype('float32') / 255.0)[..., None]

CLASS_NAMES = ['T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
               'Sandale', 'Chemise', 'Basket', 'Sac', 'Bottine']
NUM_CLASSES = 10
INPUT_SHAPE = (28, 28, 1)

# Split train / validation (85 / 15), stratifie
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.15, random_state=SEED, stratify=y_full)

# One-hot
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f"Train {X_train.shape} | Val {X_val.shape} | Test {X_test.shape}")

In [ ]:
# Apercu de quelques exemples
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i].squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[int(y_train[i])], fontsize=9)
    ax.axis('off')
plt.suptitle('Exemples Fashion-MNIST', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Hyperparametres communs a tout le TP
EPOCHS     = 15
BATCH_SIZE = 64

### Les trois briques reutilisables

- **`build_cnn(norm=..., he_init=...)`** : une seule architecture (3 blocs Conv + GAP),
  on choisit la normalisation et l'initialisation. Remplace les 5 fonctions `build_*` du TP d'origine.
- **`train(model, loss, ...)`** : compile + entraine + chronometre, avec Early Stopping systematique.
- **`evaluate(model, ...)`** : accuracy, **macro-F1** et **recall des classes minoritaires**.

In [ ]:
# Brique 1 : une architecture, parametree par la normalisation et l'init
def build_cnn(norm=None, he_init=False, dense_units=128, dropout=0.3,
              input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    """CNN 3 blocs (32, 64, 128) + GlobalAveragePooling.

    norm    : None | 'batch' | 'layer' | 'group'
    he_init : True -> 'he_normal' (adapte a ReLU), sinon 'glorot_uniform'
    """
    init = 'he_normal' if he_init else 'glorot_uniform'

    def normalization():
        if norm == 'batch': return layers.BatchNormalization()
        if norm == 'layer': return layers.LayerNormalization()
        if norm == 'group': return layers.GroupNormalization(groups=8)
        return layers.Activation('linear')          # aucun effet -> "sans normalisation"

    inputs = keras.Input(shape=input_shape)
    x = inputs
    for filters in (32, 64, 128):
        for _ in range(2):                          # 2 convolutions par bloc
            x = layers.Conv2D(filters, 3, padding='same', kernel_initializer=init)(x)
            x = normalization()(x)
            x = layers.Activation('relu')(x)
        if filters < 128:                           # pas de pooling avant le GAP final
            x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, kernel_initializer=init)(x)
    x = normalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inputs, outputs)

build_cnn(norm='batch').summary()

In [ ]:
# Brique 2 : entrainement (compile + fit + chrono), Early Stopping systematique
def train(model, loss, optimizer='adam', train_data=None, val_data=None,
          epochs=EPOCHS, patience=5, extra_callbacks=None):
    model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
    cbs = [EarlyStopping(monitor='val_loss', patience=patience,
                         restore_best_weights=True, verbose=0)]
    if extra_callbacks:
        cbs += extra_callbacks

    t0 = time.time()
    if isinstance(train_data, tf.data.Dataset):
        h = model.fit(train_data, validation_data=val_data,
                      epochs=epochs, callbacks=cbs, verbose=1)
    else:
        h = model.fit(train_data[0], train_data[1], validation_data=val_data,
                      epochs=epochs, batch_size=BATCH_SIZE, callbacks=cbs, verbose=1)
    h.train_time = time.time() - t0
    print(f"  -> temps : {h.train_time:.1f}s")
    return h


# Brique 3 : evaluation (accuracy + macro-F1 + recall des minoritaires)
def evaluate(model, X, y_onehot, y_int, title="", minority=None):
    loss, acc = model.evaluate(X, y_onehot, verbose=0)
    y_pred = np.argmax(model.predict(X, verbose=0), axis=1)
    y_int = np.asarray(y_int).flatten()

    res = {'title': title, 'loss': loss, 'accuracy': acc,
           'macro_f1': f1_score(y_int, y_pred, average='macro')}
    line = f"{title:<32} acc={acc:.4f}  macroF1={res['macro_f1']:.4f}"
    if minority is not None:
        rec = recall_score(y_int, y_pred, average=None)
        res['recall_min'] = float(np.mean([rec[c] for c in minority]))
        line += f"  recall(min)={res['recall_min']:.4f}"
    print(line)
    return res

In [ ]:
# Helpers d'affichage
def plot_history(history, title=""):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
    a1.plot(history.history['loss'], label='train')
    a1.plot(history.history['val_loss'], '--', label='val')
    a1.set_title(f'Loss - {title}'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
    a2.plot(history.history['accuracy'], label='train')
    a2.plot(history.history['val_accuracy'], '--', label='val')
    a2.set_title(f'Accuracy - {title}'); a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def compare(results, title='Comparaison des modeles'):
    rows = sorted(results, key=lambda r: r['accuracy'], reverse=True)
    cols = ['accuracy', 'macro_f1'] + (['recall_min'] if any('recall_min' in r for r in rows) else [])
    header = f"{'Modele':<32}" + ''.join(f"{c:>12}" for c in cols)
    print(header); print('-' * len(header))
    for r in rows:
        print(f"{r['title']:<32}" + ''.join(f"{r.get(c, float('nan')):>12.4f}" for c in cols))

    titles = [r['title'] for r in rows][::-1]
    accs   = [r['accuracy'] for r in rows][::-1]
    plt.figure(figsize=(9, 0.6 * len(rows) + 1))
    plt.barh(titles, accs, color='#6B1E2C')
    for i, a in enumerate(accs):
        plt.text(a + 0.005, i, f'{a:.3f}', va='center', fontweight='bold')
    plt.xlim(0, 1); plt.xlabel('Test accuracy'); plt.title(title)
    plt.tight_layout(); plt.show()

---
# PARTIE A - Fonctions de cout pour donnees desequilibrees (45 min)

On simule un desequilibre realiste : on ne garde que **10 %** des exemples des classes
**0 (T-shirt)** et **6 (Chemise)**. On compare alors trois fonctions de cout sur ce probleme.

In [ ]:
# Constantes du desequilibre (une seule source de verite)
MINORITY   = [0, 6]     # T-shirt, Chemise
KEEP_RATIO = 0.10       # on garde 10 % de ces classes

def make_imbalanced(X, y, minority=MINORITY, keep_ratio=KEEP_RATIO, seed=SEED):
    """Retire (1 - keep_ratio) des exemples des classes minoritaires."""
    rng = np.random.default_rng(seed)
    keep = np.ones(len(y), dtype=bool)
    yf = np.asarray(y).flatten()
    for c in minority:
        idx = np.where(yf == c)[0]
        drop = rng.choice(idx, int(len(idx) * (1 - keep_ratio)), replace=False)
        keep[drop] = False
    return X[keep], y[keep]

# Memes classes minoritaires et meme ratio pour train / val / test
X_train_imb, y_train_imb = make_imbalanced(X_train, y_train)
X_val_imb,   y_val_imb   = make_imbalanced(X_val,   y_val)
X_test_imb,  y_test_imb  = make_imbalanced(X_test,  y_test)

y_train_imb_cat = to_categorical(y_train_imb, NUM_CLASSES)
y_val_imb_cat   = to_categorical(y_val_imb,   NUM_CLASSES)
y_test_imb_cat  = to_categorical(y_test_imb,  NUM_CLASSES)

# Visualisation de la distribution
u, c = np.unique(y_train_imb, return_counts=True)
plt.figure(figsize=(10, 3.5))
colors = ['#6B1E2C' if k in MINORITY else '#9B3A4E' for k in u]
plt.bar([CLASS_NAMES[k] for k in u], c, color=colors)
plt.ylabel("exemples"); plt.title("Distribution train - desequilibre"); plt.xticks(rotation=30)
plt.tight_layout(); plt.show()
print(f"Train desequilibre : {len(X_train_imb)} exemples (T-shirt={c[0]}, Pull={c[2]})")

> **Pourquoi un train ET un test desequilibres ?** Pour mesurer ce qui nous interesse
> vraiment : la capacite a bien classer les **classes rares**. On regardera donc le
> **recall des minoritaires**, pas seulement l'accuracy globale.

### A.1 - Les trois fonctions de cout

- **Cross-Entropy** : reference.
- **Focal Loss** : `FL = -alpha * (1 - p_t)^gamma * log(p_t)` -> reduit le poids des exemples faciles.
- **Weighted CE** : pondere chaque classe par l'inverse de sa frequence.

In [ ]:
# Focal Loss
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce = -y_true * tf.math.log(y_pred)               # cross-entropy par classe
        weight = alpha * tf.pow(1 - y_pred, gamma)       # facteur de focalisation
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))
    return loss_fn


# Weighted Cross-Entropy
def weighted_crossentropy(class_weights):
    w = tf.constant(np.asarray(class_weights), dtype=tf.float32)
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        sample_w = tf.reduce_sum(y_true * w, axis=-1)     # poids de la vraie classe
        return tf.reduce_mean(sample_w * ce)
    return loss_fn


# Verification : Focal(gamma=0) doit redonner (a alpha pres) la Cross-Entropy
yp = tf.constant([[0.9, 0.05, 0.05], [0.1, 0.8, 0.1]])
yt = tf.constant([[1., 0., 0.], [0., 1., 0.]])
ce_ref = tf.reduce_mean(tf.reduce_sum(-yt * tf.math.log(yp), axis=-1))
print(f"CE de reference        : {ce_ref:.4f}")
print(f"Focal(gamma=0, alpha=1): {focal_loss(0.0, 1.0)(yt, yp):.4f}  (doit etre egal)")
print(f"Focal(gamma=2)         : {focal_loss(2.0, 0.25)(yt, yp):.4f}")

In [ ]:
# Poids inversement proportionnels a la frequence (pour la Weighted CE)
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced',
                                     classes=np.arange(NUM_CLASSES),
                                     y=y_train_imb.flatten())
for i, w in enumerate(class_weights):
    print(f"  {CLASS_NAMES[i]:>10} : {w:.2f}")

### A.2 - Entrainement comparatif

Meme architecture (`build_cnn(norm='batch')`), meme optimiseur : seule la **loss** change.

In [ ]:
losses_A = {
    'Cross-Entropy': 'categorical_crossentropy',
    'Focal Loss (g=2)': focal_loss(gamma=2.0, alpha=0.25),
    'Weighted CE': weighted_crossentropy(class_weights),
}

histories_A, models_A = {}, {}
for name, loss in losses_A.items():
    print(f"\n=== {name} ===")
    m = build_cnn(norm='batch')
    histories_A[name] = train(m, loss,
                              train_data=(X_train_imb, y_train_imb_cat),
                              val_data=(X_val_imb, y_val_imb_cat))
    models_A[name] = m

In [ ]:
# Courbes de validation comparees
plt.figure(figsize=(8, 4))
for name, h in histories_A.items():
    plt.plot(h.history['val_accuracy'], label=name, linewidth=2)
plt.xlabel('epoch'); plt.ylabel('val accuracy'); plt.legend(); plt.grid(alpha=.3)
plt.title('Partie A - convergence selon la loss'); plt.tight_layout(); plt.show()

In [ ]:
# Evaluation sur le TEST desequilibre : on regarde le recall des minoritaires
results_A = [evaluate(models_A[n], X_test_imb, y_test_imb_cat, y_test_imb, n, minority=MINORITY)
             for n in losses_A]
compare(results_A, title='Partie A - donnees desequilibrees')

### Question A - a rediger

1. Pourquoi l'**accuracy globale** est-elle un mauvais critere ici ? Que regarde-t-on a la place ?
2. Comparez le **recall des minoritaires** (T-shirt, Chemise) entre les trois loss. Que constatez-vous ?
3. Pourquoi les **valeurs de loss** ne sont-elles pas comparables entre CE, Focal et Weighted CE ?
4. Quelle loss offre le meilleur **compromis** recall(min) / accuracy globale ?
5. *Bonus* : que vaut la Focal Loss quand `gamma = 0` ? Verifiez-le.

*Votre reponse :*

...

---
# PARTIE B - Normalisation : sans / BatchNorm / LayerNorm / GroupNorm (45 min)

On revient au dataset **equilibre** pour isoler l'effet de la normalisation.
Grace a `build_cnn(norm=...)`, les 4 variantes partagent **exactement** la meme architecture.

In [ ]:
norm_variants = {
    'Sans norm':       None,
    'BatchNorm':       'batch',
    'LayerNorm':       'layer',
    'GroupNorm (g=8)': 'group',
}

histories_B, models_B = {}, {}
for name, kind in norm_variants.items():
    print(f"\n=== {name} ===")
    m = build_cnn(norm=kind)
    print(f"  parametres : {m.count_params():,}")
    histories_B[name] = train(m, 'categorical_crossentropy',
                              train_data=(X_train, y_train_cat),
                              val_data=(X_val, y_val_cat), patience=8)
    models_B[name] = m

In [ ]:
# Convergence comparee
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in histories_B.items():
    a1.plot(h.history['val_accuracy'], label=name, linewidth=2)
    a2.plot(h.history['val_loss'],     label=name, linewidth=2)
a1.set_title('Val accuracy'); a1.set_xlabel('epoch'); a1.legend(); a1.grid(alpha=.3)
a2.set_title('Val loss');     a2.set_xlabel('epoch'); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# Evaluation sur le test (equilibre) + temps et nombre de parametres
results_B = []
for name in norm_variants:
    r = evaluate(models_B[name], X_test, y_test_cat, y_test, name)
    r['time']   = histories_B[name].train_time
    r['params'] = models_B[name].count_params()
    results_B.append(r)

compare(results_B, title='Partie B - normalisation')
print(f"\n{'Modele':<18}{'params':>12}{'temps (s)':>12}")
for r in results_B:
    print(f"{r['title']:<18}{r['params']:>12,}{r['time']:>12.1f}")

### Question B - a rediger

1. Quelle normalisation donne la **meilleure accuracy** test ?
2. Laquelle **converge le plus vite** (epochs pour atteindre ~70 % de val accuracy) ?
3. Observe-t-on un effet **regularisant** de la BatchNorm (gap train/val plus faible) ?
4. Quand recommanderiez-vous **GroupNorm** plutot que BatchNorm ?
5. Pourquoi **LayerNorm** est-elle moins adaptee aux CNN ?

*Votre reponse :*

...

---
# PARTIE C - Le modele optimise + etude d'ablation (45 min)

On combine les meilleures techniques :
**He init + BatchNorm + AdamW + Label Smoothing + Data Augmentation**,
puis on retire **une technique a la fois** pour mesurer sa contribution.

In [ ]:
# Data augmentation (couches Keras) + apercu
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(X_train[0].squeeze(), cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
for i in range(1, 6):
    aug = data_augmentation(X_train[:1], training=True)[0]
    axes[i].imshow(aug.numpy().squeeze(), cmap='gray'); axes[i].set_title(f'aug {i}'); axes[i].axis('off')
plt.suptitle('Data augmentation', fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
# Label smoothing : illustration sur un label
eps = 0.1
hard = np.zeros(NUM_CLASSES); hard[3] = 1.0                 # classe "Robe"
smooth = (1 - eps) * hard + eps / NUM_CLASSES
print("Label dur   :", hard)
print("Label lisse :", np.round(smooth, 3), "  (somme =", smooth.sum(), ")")

In [ ]:
# Dataset augmente (tf.data) : l'augmentation ne s'applique qu'au train
AUTOTUNE = tf.data.AUTOTUNE
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train_cat))
            .shuffle(10000).batch(BATCH_SIZE)
            .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))
val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val_cat))
          .batch(BATCH_SIZE).prefetch(AUTOTUNE))
print("Datasets tf.data prets.")

### C.1 - Ablation : un seul tableau, une seule boucle

Chaque ligne enleve **une** technique du modele complet.

In [ ]:
ls_loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
ce_loss = 'categorical_crossentropy'

def adamw():  return keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)
def adam():   return keras.optimizers.Adam(learning_rate=1e-3)

# config : (optimiseur, loss, augmentation ?)
ablation = {
    'Optimise (complet)':       (adamw, ls_loss, True),
    'sans Label Smoothing':     (adamw, ce_loss, True),
    'sans Augmentation':        (adamw, ls_loss, False),
    'Adam (sans weight decay)': (adam,  ls_loss, True),
}

results_C, history_opt, model_optimized = [], None, None
for name, (opt, loss, use_aug) in ablation.items():
    print(f"\n=== {name} ===")
    m = build_cnn(norm='batch', he_init=True, dense_units=256, dropout=0.4)
    tr  = train_ds if use_aug else (X_train, y_train_cat)
    val = val_ds   if use_aug else (X_val, y_val_cat)
    h = train(m, loss, optimizer=opt(), train_data=tr, val_data=val, patience=10,
              extra_callbacks=[ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                 patience=4, min_lr=1e-6, verbose=0)])
    results_C.append(evaluate(m, X_test, y_test_cat, y_test, name))
    if name == 'Optimise (complet)':
        history_opt, model_optimized = h, m

In [ ]:
plot_history(history_opt, title="Modele optimise complet")
compare(results_C, title='Partie C - ablation')

### Question C - a rediger

1. Quelle technique apporte le **plus grand gain** d'accuracy ?
2. Le **Label Smoothing** a-t-il un impact ? Regardez aussi le **gap train/val**.
3. **AdamW** bat-il **Adam** ici ? Pourquoi (weight decay decouple) ?
4. La **Data Augmentation** est-elle utile sur Fashion-MNIST ? Quand serait-elle critique ?

*Votre reponse :*

...

---
# PARTIE D - Analyse comparative et analyse d'erreurs (45 min)

Comparaison **equitable** : la baseline utilise la **meme architecture** que tout le reste,
mais **sans aucune technique** (`build_cnn()` : pas de norm, init standard, optimiseur Adam).

In [ ]:
# Baseline equitable (meme archi, sans techniques)
print("=== Baseline (CNN nu + Adam + CE) ===")
m_base = build_cnn()                       # norm=None, he_init=False
train(m_base, 'categorical_crossentropy',
      train_data=(X_train, y_train_cat), val_data=(X_val, y_val_cat), patience=8)

best_B = max(results_B, key=lambda r: r['accuracy'])

final = [
    evaluate(m_base, X_test, y_test_cat, y_test, "Baseline (sans technique)"),
    evaluate(models_B[best_B['title']], X_test, y_test_cat, y_test, f"Meilleure norm ({best_B['title']})"),
    results_C[0],                          # modele optimise complet
]
compare(final, title='Partie D - baseline -> optimise')

gain = (final[-1]['accuracy'] - final[0]['accuracy']) * 100
print(f"\nGain total baseline -> optimise : +{gain:.2f} points d'accuracy")

### D.1 - Analyse des erreurs du modele optimise

In [ ]:
# Predictions et erreurs
y_prob = model_optimized.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = y_test.flatten()
errors = np.where(y_pred != y_true)[0]
print(f"Erreurs : {len(errors)} / {len(y_test)} ({len(errors)/len(y_test)*100:.1f} %)")

# 16 erreurs les plus confiantes (les plus "dangereuses")
worst = errors[np.argsort(-y_prob[errors].max(axis=1))][:16]
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for ax, idx in zip(axes.flat, worst):
    ax.imshow(X_test[idx].squeeze(), cmap='gray')
    ax.set_title(f"vrai {CLASS_NAMES[y_true[idx]]}\npred {CLASS_NAMES[y_pred[idx]]}"
                 f"\n{y_prob[idx].max()*100:.0f}%", fontsize=7, color='#6B1E2C')
    ax.axis('off')
plt.suptitle('Erreurs les plus confiantes', fontweight='bold'); plt.tight_layout(); plt.show()

In [ ]:
# Top 5 des confusions
cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(cm, 0)
print("Top 5 des paires les plus confondues :")
for _ in range(5):
    i, j = np.unravel_index(cm.argmax(), cm.shape)
    print(f"  {CLASS_NAMES[i]:>10} -> {CLASS_NAMES[j]:<10} : {cm[i, j]} erreurs")
    cm[i, j] = 0

In [ ]:
# Calibration : confiance sur predictions correctes vs incorrectes
conf = y_prob.max(axis=1)
ok = y_pred == y_true
plt.figure(figsize=(9, 4))
plt.hist(conf[ok],  bins=40, alpha=0.7, color='#2E7D32', label='correctes')
plt.hist(conf[~ok], bins=40, alpha=0.7, color='#C62828', label='incorrectes')
plt.xlabel('confiance (max softmax)'); plt.ylabel('nb predictions'); plt.legend(); plt.grid(alpha=.3)
plt.title('Distribution de la confiance'); plt.tight_layout(); plt.show()
print(f"Confiance moyenne - correctes   : {conf[ok].mean():.3f}")
print(f"Confiance moyenne - incorrectes : {conf[~ok].mean():.3f}")

### Question D - synthese (a rediger, 10-15 lignes)

1. Quel est le **gain total** d'accuracy baseline -> optimise ?
2. Quelles **classes** restent les plus difficiles ? Pourquoi (cf. top confusions) ?
3. Le modele est-il **bien calibre** (cf. distribution de confiance) ?
4. Trois **pistes** pour aller plus loin ?
5. La **lecon principale** de ce TP sur l'optimisation ?

*Votre reponse :*

...

---
**Fin du TP 1** - Seance suivante : *Amelioration des performances & generalisation*